# Detecção e Contagem de Pessoas com YOLOv8m — Avaliação de Desempenho

**Projeto de pesquisa** para avaliar o desempenho do modelo **YOLOv8m** na tarefa de
detecção/contagem de pessoas (classe `person` = `0` do COCO).

Além da aplicação original (contagem dentro de uma **ROI** via *tracking*), este notebook
adiciona uma camada completa de avaliação sobre um conjunto de teste rotulado em formato YOLO:

- **Precision, Recall, F1-score** (a partir do casamento por IoU predição × gabarito);
- **mAP@50** e **mAP@50-95** (via `ultralytics model.val()`);
- **Matriz de confusão** (`person` vs `background`) com *scikit-learn* + *seaborn*;
- **Tabela de resultados** por imagem (CSV), **gráficos** de análise e **relatório** textual.

### Dataset compactado (`dataset.zip`)
Para não versionar milhares de imagens no repositório, o dataset fica em **`dataset.zip`**
na raiz do projeto. O notebook o extrai automaticamente para `dataset/` (se ainda não
existir) e localiza o `data.yaml` em qualquer nível — funciona tanto com
`dataset/data.yaml` quanto com `dataset/Person Dataset/data.yaml`, etc.

In [1]:
# Dependências (descomente na primeira execução, ex.: Google Colab):
# !pip install ultralytics scikit-learn seaborn matplotlib pandas opencv-python pyyaml

## 1. Imports e configuração

Todas as constantes do experimento ficam centralizadas aqui para facilitar a reprodução.

In [2]:
import os
import time
import zipfile
import warnings
from pathlib import Path

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from ultralytics import YOLO

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ----------------------------------------------------------------------
# Caminhos (a raiz do projeto é a pasta onde o notebook está sendo executado)
# ----------------------------------------------------------------------
BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "dataset"        # destino da extração
ZIP_PATH = BASE_DIR / "dataset.zip"       # dataset compactado (não versionar a pasta)
RESULTS_DIR = BASE_DIR / "results"        # saídas (CSV, gráficos, relatório)

# ----------------------------------------------------------------------
# Parâmetros do experimento
# ----------------------------------------------------------------------
WEIGHTS = "yolov8m.pt"   # baixado automaticamente pela ultralytics
PERSON_CLASS_ID = 0      # "person" no COCO (usado nas PREDIÇÕES do modelo)
CONF = 0.5               # confiança mínima na avaliação por imagem
IOU_MATCH = 0.5          # IoU mínimo para casar predição x gabarito (matriz de confusão)
IMGSZ = 1280             # tamanho de entrada da inferência

# ROI da aplicação (contagem em fila) — usada por contar_pessoas()
ROI_X1, ROI_Y1, ROI_X2, ROI_Y2 = 100, 100, 900, 700

# Garante a pasta de saída
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Pasta de resultados:", RESULTS_DIR)

Pasta de resultados: c:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\results


## 2. Carregamento do modelo

`carregar_modelo()` apenas instancia o YOLOv8m. O peso `.pt` é baixado automaticamente
na primeira execução.

In [3]:
def carregar_modelo(weights: str = WEIGHTS) -> YOLO:
    """Carrega e retorna o modelo YOLOv8m.

    O arquivo de pesos (.pt) é baixado automaticamente pela ultralytics
    na primeira vez que o modelo é instanciado.
    """
    print(f"Carregando modelo: {weights}")
    return YOLO(weights)

## 3. Contagem por ROI (aplicação) — função original mantida

Esta é a função do projeto original, preservada como o caso de uso de **aplicação**:
ela usa *tracking* (`model.track`) para não contar a mesma pessoa duas vezes e conta
apenas quem está dentro da Região de Interesse (ROI). É **independente** da avaliação
do dataset (que usa detecção quadro a quadro, sem tracking).

In [4]:
def contar_pessoas(imagem, modelo, roi=(ROI_X1, ROI_Y1, ROI_X2, ROI_Y2)):
    """[APLICACAO] Conta pessoas dentro de uma ROI usando rastreamento.

    Parametros
    ----------
    imagem : caminho/np.ndarray/stream aceito pela ultralytics.
    modelo : instancia YOLO ja carregada.
    roi    : (x1, y1, x2, y2) da Regiao de Interesse, em pixels.

    Retorna
    -------
    (quantidade, frame_saida, estatisticas)
    """
    x1_roi, y1_roi, x2_roi, y2_roi = roi
    inicio = time.time()

    # Rastreamento persistente apenas da classe "person"
    resultados = modelo.track(
        source=imagem,
        classes=[PERSON_CLASS_ID],
        persist=True,
        conf=0.55,
        imgsz=IMGSZ,
        verbose=False,
    )

    tempo_execucao = time.time() - inicio

    # Frame anotado pela ultralytics + desenho da ROI
    frame_saida = resultados[0].plot()
    cv2.rectangle(frame_saida, (x1_roi, y1_roi), (x2_roi, y2_roi), (255, 0, 0), 2)

    ids_contados = set()
    confiancas = []

    # Conta IDs unicos cujo centro cai dentro da ROI
    if resultados[0].boxes.id is not None:
        ids = resultados[0].boxes.id.cpu().numpy()
        boxes = resultados[0].boxes.xyxy.cpu().numpy()
        confs = resultados[0].boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(boxes, ids, confs):
            bx1, by1, bx2, by2 = box
            cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
            if x1_roi <= cx <= x2_roi and y1_roi <= cy <= y2_roi:
                ids_contados.add(int(track_id))
                confiancas.append(float(conf))

    quantidade = len(ids_contados)
    estatisticas = {
        "tempo_execucao": tempo_execucao,
        "quantidade": quantidade,
        "confiancas": confiancas,
    }
    return quantidade, frame_saida, estatisticas

## 4. Preparação do Dataset

`preparar_dataset()` deixa o dataset pronto sem precisar versionar milhares de arquivos:

1. procura `dataset.zip` na raiz do projeto;
2. se a pasta `dataset/` **não** existir, extrai o zip para `dataset/`;
3. usa `os.walk()` para localizar o `data.yaml` **em qualquer nível** da árvore extraída;
4. retorna o **caminho absoluto** do `data.yaml`.

As funções auxiliares `resolver_split()` e `id_pessoa_no_dataset()` interpretam esse
`data.yaml` (formato Roboflow/Ultralytics) para descobrir qual pasta de imagens avaliar e
qual é o índice da classe *person* no gabarito — assim o notebook lida com estruturas como
`dataset/train|valid|test + data.yaml` ou `dataset/Person Dataset/...` sem alterar código.

In [5]:
def preparar_dataset(zip_path: Path = ZIP_PATH, dataset_dir: Path = DATASET_DIR) -> Path:
    """Extrai dataset.zip (se necessario) e retorna o caminho absoluto do data.yaml.

    1. Se a pasta dataset/ nao existir, extrai dataset.zip para dataset/.
    2. Procura recursivamente (os.walk) por 'data.yaml' dentro de dataset/.
    3. Retorna o caminho absoluto do data.yaml encontrado.
    """
    # 1 e 2: extrai apenas se a pasta ainda nao existe
    if not dataset_dir.exists():
        if not zip_path.exists():
            raise FileNotFoundError(
                f"Nao encontrei '{zip_path.name}' na raiz do projeto nem a pasta "
                f"'{dataset_dir.name}/'. Coloque o dataset compactado como {zip_path}."
            )
        print(f"Extraindo {zip_path.name} -> {dataset_dir} ...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(dataset_dir)
        print("Extracao concluida.")
    else:
        print(f"Pasta '{dataset_dir}' ja existe (extracao ignorada).")

    # 3: localiza o data.yaml em qualquer nivel da estrutura extraida
    for raiz, _dirs, arquivos in os.walk(dataset_dir):
        if "data.yaml" in arquivos:
            caminho = (Path(raiz) / "data.yaml").resolve()
            print(f"data.yaml encontrado: {caminho}")
            return caminho

    raise FileNotFoundError(
        f"'data.yaml' nao encontrado dentro de '{dataset_dir}'. "
        "Verifique o conteudo do dataset.zip."
    )


def _resolver_dir(valor: str, base: Path, raiz: Path):
    """Tenta resolver um caminho de split em diferentes bases.

    Cobre os formatos comuns de data.yaml: caminho absoluto, relativo ao campo
    'path' (raiz), relativo ao proprio yaml (base) e o padrao do Roboflow, que usa
    '../test/images' mesmo com as pastas dentro do diretorio do yaml.
    """
    candidatos = []
    p = Path(valor)
    if p.is_absolute():
        candidatos.append(p)

    # variante "limpa": sem ./ inicial e sem ../ no comeco
    limpo = valor.lstrip("./")
    while limpo.startswith("../"):
        limpo = limpo[3:]

    candidatos += [raiz / valor, base / valor, raiz / limpo, base / limpo]
    for c in candidatos:
        c = Path(os.path.normpath(str(c)))
        if c.exists():
            return c
    return None


def resolver_split(data_yaml: Path, preferencia=("test", "val", "valid", "train")):
    """Resolve (split, images_dir, labels_dir) do melhor split disponivel no data.yaml.

    Aceita caminhos relativos ao proprio yaml ou ao campo 'path'. A pasta de labels
    e derivada trocando 'images' por 'labels' (convencao YOLO)."""
    cfg = yaml.safe_load(Path(data_yaml).read_text(encoding="utf-8"))
    base = Path(data_yaml).parent
    raiz = base
    if cfg.get("path"):
        cand = _resolver_dir(str(cfg["path"]), base, base)
        if cand:
            raiz = cand

    for chave in preferencia:
        valor = cfg.get(chave)
        if not valor:
            continue
        img_dir = _resolver_dir(str(valor), base, raiz)
        if img_dir and img_dir.exists():
            labels_dir = Path(str(img_dir).replace("images", "labels"))
            return chave, img_dir, labels_dir

    raise FileNotFoundError(
        "Nenhum split valido (test/val/train) encontrado/existente no data.yaml."
    )


def id_pessoa_no_dataset(data_yaml: Path) -> int:
    """Indice da classe 'person' segundo o data.yaml (0 se nao encontrar)."""
    cfg = yaml.safe_load(Path(data_yaml).read_text(encoding="utf-8"))
    names = cfg.get("names")
    if isinstance(names, dict):
        for k, v in names.items():
            if str(v).strip().lower() == "person":
                return int(k)
    elif isinstance(names, list):
        for i, v in enumerate(names):
            if str(v).strip().lower() == "person":
                return i
    return 0

## 5. Utilitários de rótulos (formato YOLO)

Funções para listar imagens e ler o gabarito (`.txt`): contagem de pessoas e conversão
das caixas de YOLO-normalizado para `xyxy` em pixels.

In [6]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def listar_imagens(images_dir: Path):
    """Lista (ordenado) os arquivos de imagem da pasta."""
    images_dir = Path(images_dir)
    return sorted(p for p in images_dir.glob("*") if p.suffix.lower() in IMG_EXTS)


def contar_pessoas_no_label(label_path: Path, classe_id: int = PERSON_CLASS_ID) -> int:
    """Conta quantas pessoas (classe `classe_id`) existem no rotulo YOLO da imagem."""
    label_path = Path(label_path)
    if not label_path.exists():
        return 0
    total = 0
    for linha in label_path.read_text().splitlines():
        linha = linha.strip()
        if linha and int(float(linha.split()[0])) == classe_id:
            total += 1
    return total


def ler_caixas_gt(label_path: Path, img_w: int, img_h: int,
                  classe_id: int = PERSON_CLASS_ID):
    """Le as caixas de gabarito (classe `classe_id`) e converte de YOLO-normalizado
    para coordenadas xyxy em pixels."""
    label_path = Path(label_path)
    caixas = []
    if not label_path.exists():
        return caixas
    for linha in label_path.read_text().splitlines():
        partes = linha.split()
        if len(partes) < 5 or int(float(partes[0])) != classe_id:
            continue
        xc, yc, w, h = map(float, partes[1:5])
        x1 = (xc - w / 2) * img_w
        y1 = (yc - h / 2) * img_h
        x2 = (xc + w / 2) * img_w
        y2 = (yc + h / 2) * img_h
        caixas.append([x1, y1, x2, y2])
    return caixas

## 6. Avaliação do dataset (tabela + CSV)

`avaliar_dataset()` roda a inferência (detecção, sem tracking) em cada imagem do split
escolhido e coleta, por imagem: contagem **real** × **detectada**, **erro absoluto**,
**tempo de execução** e **confiança média**. Exporta a tabela para `results/resultados.csv`
e guarda as caixas (predição × gabarito) para a matriz de confusão.

In [7]:
def iou(boxA, boxB) -> float:
    """Intersection over Union entre duas caixas xyxy."""
    xa, ya = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xb, yb = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0.0, xb - xa) * max(0.0, yb - ya)
    areaA = max(0.0, boxA[2] - boxA[0]) * max(0.0, boxA[3] - boxA[1])
    areaB = max(0.0, boxB[2] - boxB[0]) * max(0.0, boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0


def avaliar_dataset(modelo, images_dir, labels_dir, gt_person_id: int = PERSON_CLASS_ID,
                    conf: float = CONF):
    """Avalia o modelo imagem a imagem no split informado.

    Retorna
    -------
    df            : pandas.DataFrame com a tabela de resultados.
    agregados     : dict com tempo medio, confianca media, n_imagens, MAE.
    pares_para_cm : lista (caixas_pred, scores_pred, caixas_gt) por imagem.
    """
    images_dir, labels_dir = Path(images_dir), Path(labels_dir)
    imagens = listar_imagens(images_dir)

    linhas = []
    pares_para_cm = []

    for img_path in imagens:
        label_path = labels_dir / (img_path.stem + ".txt")

        # ---- Inferencia (cronometrada) ----
        inicio = time.perf_counter()
        resultados = modelo.predict(
            source=str(img_path),
            classes=[PERSON_CLASS_ID],
            conf=conf,
            imgsz=IMGSZ,
            verbose=False,
        )
        tempo_execucao = time.perf_counter() - inicio

        r = resultados[0]
        img_h, img_w = r.orig_shape  # (altura, largura)

        caixas_pred = r.boxes.xyxy.cpu().numpy().tolist() if r.boxes is not None else []
        scores_pred = r.boxes.conf.cpu().numpy().tolist() if r.boxes is not None else []

        # ---- Metricas por imagem ----
        qtd_detectada = len(caixas_pred)
        qtd_real = contar_pessoas_no_label(label_path, gt_person_id)
        conf_media = float(np.mean(scores_pred)) if scores_pred else 0.0
        erro_abs = abs(qtd_real - qtd_detectada)

        caixas_gt = ler_caixas_gt(label_path, img_w, img_h, gt_person_id)
        pares_para_cm.append((caixas_pred, scores_pred, caixas_gt))

        linhas.append(
            {
                "Imagem": img_path.name,
                "Quantidade real": qtd_real,
                "Quantidade detectada": qtd_detectada,
                "Erro absoluto": erro_abs,
                "Tempo de execução": round(tempo_execucao, 4),
                "Confiança média": round(conf_media, 4),
            }
        )

    df = pd.DataFrame(linhas)

    # ---- Exporta a tabela ----
    csv_path = RESULTS_DIR / "resultados.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"Tabela salva em: {csv_path}")

    todas_confiancas = [s for (_, scores, _) in pares_para_cm for s in scores]
    agregados = {
        "n_imagens": int(len(df)),
        "tempo_medio": float(df["Tempo de execução"].mean()) if len(df) else 0.0,
        "confianca_media": float(np.mean(todas_confiancas)) if todas_confiancas else 0.0,
        "erro_medio_absoluto": float(df["Erro absoluto"].mean()) if len(df) else 0.0,
    }
    return df, agregados, pares_para_cm

## 7. Métricas de detecção (mAP via ultralytics)

`avaliar_metricas_deteccao()` usa a validação oficial da ultralytics (`model.val`) sobre
o `data.yaml` do dataset para obter **mAP@50** e **mAP@50-95** (e também P/R no esquema de
detecção com varredura de confiança). É o padrão da literatura para avaliar detectores.

> Observação: assume-se um dataset de pessoa(s) cujo índice de `person` corresponde ao do
> COCO (`0`). Para datasets multi-classe, ajuste `classes`/o mapeamento conforme necessário.

In [8]:
def avaliar_metricas_deteccao(modelo, data_yaml, split: str = "val",
                              imgsz: int = IMGSZ) -> dict:
    """Roda model.val() no split indicado e retorna mAP50, mAP50-95 e P/R."""
    split = "val" if split == "valid" else split   # ultralytics usa 'val'
    metrics = modelo.val(
        data=str(data_yaml),
        split=split,
        classes=[PERSON_CLASS_ID],
        imgsz=imgsz,
        verbose=False,
    )
    return {
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "precision_val": float(metrics.box.mp),
        "recall_val": float(metrics.box.mr),
    }

## 8. Matriz de confusão (person vs background)

`gerar_matriz_confusao()` casa predições e gabarito por **IoU ≥ 0.5** (guloso, da maior
para a menor confiança): par casado → **TP**; gabarito sem par → **FN**; predição sem par
→ **FP**. Monta a matriz com `sklearn.metrics.confusion_matrix`, plota com *seaborn* e
salva em `results/confusion_matrix.png`. Daí derivam **Precision/Recall/F1** coerentes
com a matriz.

In [9]:
def gerar_matriz_confusao(pares_para_cm, results_dir: Path = RESULTS_DIR,
                          iou_match: float = IOU_MATCH) -> dict:
    """Constroi a matriz de confusao person vs background por casamento de IoU."""
    y_true, y_pred = [], []

    for caixas_pred, scores_pred, caixas_gt in pares_para_cm:
        ordem = list(np.argsort(scores_pred)[::-1]) if scores_pred else []
        gt_usado = [False] * len(caixas_gt)

        for i in ordem:
            melhor_iou, melhor_j = 0.0, -1
            for j, gt in enumerate(caixas_gt):
                if gt_usado[j]:
                    continue
                atual = iou(caixas_pred[i], gt)
                if atual > melhor_iou:
                    melhor_iou, melhor_j = atual, j
            if melhor_iou >= iou_match and melhor_j >= 0:
                gt_usado[melhor_j] = True
                y_true.append(1); y_pred.append(1)   # TP
            else:
                y_true.append(0); y_pred.append(1)   # FP

        for usado in gt_usado:
            if not usado:
                y_true.append(1); y_pred.append(0)   # FN

    if not y_true:
        print("[AVISO] Sem predicoes/gabaritos para a matriz de confusao.")
        return {"TP": 0, "FP": 0, "FN": 0, "precision": 0.0, "recall": 0.0, "f1": 0.0}

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["background", "person"],
        yticklabels=["background", "person"],
    )
    plt.xlabel("Predito"); plt.ylabel("Real")
    plt.title("Matriz de Confusão (person vs background)")
    plt.tight_layout()
    caminho = results_dir / "confusion_matrix.png"
    plt.savefig(caminho, dpi=150)
    plt.show()
    print(f"Matriz de confusão salva em: {caminho}")

    return {
        "TP": int(tp), "FP": int(fp), "FN": int(fn),
        "precision": precision, "recall": recall, "f1": f1,
    }

## 9. Gráficos de análise

`gerar_graficos()` salva quatro figuras em `results/`: reais × detectadas,
distribuição das confianças, tempo de execução por imagem e histograma das detecções.

In [10]:
def gerar_graficos(df, pares_para_cm=None, results_dir: Path = RESULTS_DIR):
    """Gera e salva os 4 graficos de analise."""
    if df.empty:
        print("[AVISO] DataFrame vazio - nada para plotar.")
        return

    # 1) Pessoas reais x detectadas
    plt.figure(figsize=(6, 6))
    maxv = int(max(df["Quantidade real"].max(), df["Quantidade detectada"].max(), 1))
    plt.scatter(df["Quantidade real"], df["Quantidade detectada"], alpha=0.7)
    plt.plot([0, maxv], [0, maxv], "r--", label="ideal (y = x)")
    plt.xlabel("Pessoas reais"); plt.ylabel("Pessoas detectadas")
    plt.title("Reais x Detectadas"); plt.legend(); plt.tight_layout()
    plt.savefig(results_dir / "real_vs_detectado.png", dpi=150); plt.show()

    # 2) Distribuicao das confiancas
    confs = [s for (_, scores, _) in pares_para_cm for s in scores] if pares_para_cm else []
    plt.figure(figsize=(7, 4))
    if confs:
        plt.hist(confs, bins=20, color="#3b82f6", edgecolor="black")
    plt.xlabel("Confiança"); plt.ylabel("Frequência")
    plt.title("Distribuição das Confianças"); plt.tight_layout()
    plt.savefig(results_dir / "distribuicao_confiancas.png", dpi=150); plt.show()

    # 3) Tempo de execucao por imagem
    plt.figure(figsize=(8, 4))
    plt.bar(range(len(df)), df["Tempo de execução"], color="#10b981")
    plt.axhline(df["Tempo de execução"].mean(), color="red", linestyle="--",
                label=f"média = {df['Tempo de execução'].mean():.3f}s")
    plt.xlabel("Imagem (índice)"); plt.ylabel("Tempo (s)")
    plt.title("Tempo de Execução por Imagem"); plt.legend(); plt.tight_layout()
    plt.savefig(results_dir / "tempo_execucao.png", dpi=150); plt.show()

    # 4) Histograma das deteccoes
    plt.figure(figsize=(7, 4))
    maxd = int(df["Quantidade detectada"].max())
    plt.hist(df["Quantidade detectada"], bins=range(0, maxd + 2),
             color="#f59e0b", edgecolor="black", align="left")
    plt.xlabel("Pessoas detectadas"); plt.ylabel("Nº de imagens")
    plt.title("Histograma das Detecções"); plt.tight_layout()
    plt.savefig(results_dir / "histograma_deteccoes.png", dpi=150); plt.show()

    print(f"Gráficos salvos em: {results_dir}")

## 10. Relatório consolidado

`gerar_relatorio()` monta o resumo final e o grava em `results/relatorio.txt`
através de `salvar_relatorio()`.

In [11]:
def salvar_relatorio(texto: str, caminho: Path = RESULTS_DIR / "relatorio.txt") -> None:
    """Grava o texto do relatorio em disco."""
    Path(caminho).write_text(texto, encoding="utf-8")
    print(f"Relatório salvo em: {caminho}")


def gerar_relatorio(metricas: dict, caminho: Path = RESULTS_DIR / "relatorio.txt") -> str:
    """Monta o resumo da avaliacao e salva em results/relatorio.txt."""
    linhas = [
        "=" * 52,
        " RELATÓRIO DE AVALIAÇÃO - YOLOv8m (contagem de pessoas)",
        "=" * 52,
        f"Imagens avaliadas        : {metricas.get('n_imagens', 0)}",
        f"Precision                : {metricas.get('precision', 0.0):.4f}",
        f"Recall                   : {metricas.get('recall', 0.0):.4f}",
        f"F1-score                 : {metricas.get('f1', 0.0):.4f}",
        f"mAP@50                   : {metricas.get('map50', 0.0):.4f}",
        f"mAP@50-95                : {metricas.get('map50_95', 0.0):.4f}",
        f"Tempo médio por imagem   : {metricas.get('tempo_medio', 0.0):.4f} s",
        f"Confiança média          : {metricas.get('confianca_media', 0.0):.4f}",
        f"Erro médio de contagem   : {metricas.get('erro_medio_absoluto', 0.0):.4f}",
        "=" * 52,
    ]
    texto = "\n".join(linhas)
    salvar_relatorio(texto, caminho)
    print(texto)
    return texto

## 11. Pipeline principal

Executa a avaliação de ponta a ponta: prepara o dataset a partir de `dataset.zip`,
escolhe o melhor split (test → val → train), roda a inferência por imagem, calcula as
métricas e gera todos os artefatos em `results/`.

In [12]:
# 1) Carrega o modelo
modelo = carregar_modelo()

# 2) Prepara o dataset (extrai dataset.zip se preciso) e localiza o data.yaml
data_yaml = preparar_dataset()

# 3) Resolve o split a avaliar e o indice da classe "person" no gabarito
split, images_dir, labels_dir = resolver_split(data_yaml)
gt_id = id_pessoa_no_dataset(data_yaml)
print(f"Split avaliado : {split}")
print(f"Imagens        : {images_dir}")
print(f"Rotulos        : {labels_dir}")
print(f"Classe person (gabarito) = {gt_id}")

# 4) Avaliacao por imagem (tabela + CSV)
df, agregados, pares = avaliar_dataset(modelo, images_dir, labels_dir, gt_person_id=gt_id)
try:
    display(df)
except NameError:
    print(df)

# 5) Metricas de deteccao (mAP via ultralytics) no mesmo split
metricas_val = avaliar_metricas_deteccao(modelo, data_yaml, split=split)

# 6) Matriz de confusao (Precision/Recall/F1 coerentes com a CM)
metricas_cm = gerar_matriz_confusao(pares)

# 7) Graficos
gerar_graficos(df, pares_para_cm=pares)

# 8) Relatorio consolidado
metricas = {**agregados, **metricas_val, **metricas_cm}
gerar_relatorio(metricas)

Carregando modelo: yolov8m.pt
Extraindo dataset.zip -> c:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\dataset ...
Extracao concluida.
data.yaml encontrado: C:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\dataset\data.yaml
Split avaliado : test
Imagens        : C:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\dataset\test\images
Rotulos        : C:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\dataset\test\labels
Classe person (gabarito) = 0
Tabela salva em: c:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\results\resultados.csv


,Imagem,Quantidade real,Quantidade detectada,Erro absoluto,Tempo de execução,Confiança média
0,000000000474_jpg.rf.1a8888e1446b53617ec9b39344...,1,0,1,1.6361,0.0000
1,000000001295_jpg.rf.987f2f516be2b9310e2490e6bd...,12,6,6,1.2704,0.7077
2,000000005038_jpg.rf.20a447f0e56affb0e510e02d3d...,1,1,0,1.0456,0.6581
3,000000005205_jpg.rf.17f415adc4352f42bc8ce2a542...,2,2,0,1.0490,0.7835
4,000000005620_jpg.rf.837ff448b212a4115ae81bcac2...,13,9,4,1.1261,0.7196
...,...,...,...,...,...,...
515,000000495712_jpg.rf.c32007bdeb71334d3595dfb55a...,2,1,1,1.0057,0.5593
516,000000498918_jpg.rf.16227f17f8319c7053d1199b2a...,14,7,7,1.0124,0.6345
517,000000499179_jpg.rf.dd8a9e86bc87a2c4d3bbcb9221...,1,0,1,1.0155,0.0000
518,000000499357_jpg.rf.9247152caf67441476040d568c...,5,3,2,1.0090,0.7729


Ultralytics 8.4.72  Python-3.12.6 torch-2.12.1+cpu CPU (AMD Ryzen 5 5600X 6-Core Processor)
val: Fast image access  (ping: 0.00.0 ms, read: 256.522.9 MB/s, size: 27.6 KB)
val: Scanning C:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\dataset\test\labels... 520 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 520/520 1.8Kit/s 0.3s<0.2s
val: New cache created: C:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\dataset\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 15.7s/it 8:3715.7s
                   all        520       2049       0.67      0.569       0.62      0.373
Speed: 4.0ms preprocess, 978.1ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to C:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\runs\detect\val


<Figure size 600x500 with 2 Axes>

Matriz de confusão salva em: c:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\results\confusion_matrix.png


...

<Figure size 700x400 with 1 Axes>

<Figure size 800x400 with 1 Axes>

<Figure size 700x400 with 1 Axes>

Gráficos salvos em: c:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\results
Relatório salvo em: c:\Users\Nicolas\Desktop\Projetos\PESSOAL\Contador-de-pessoas-com-YOLO\results\relatorio.txt
 RELATÓRIO DE AVALIAÇÃO - YOLOv8m (contagem de pessoas)
Imagens avaliadas        : 520
Precision                : 0.0248
Recall                   : 0.0127
F1-score                 : 0.0168
mAP@50                   : 0.6203
mAP@50-95                : 0.3727
Tempo médio por imagem   : 0.9973 s
Confiança média          : 0.7259
Erro médio de contagem   : 2.1096


'====================================================\n RELATÓRIO DE AVALIAÇÃO - YOLOv8m (contagem de pessoas)\n====================================================\nImagens avaliadas        : 520\nPrecision                : 0.0248\nRecall                   : 0.0127\nF1-score                 : 0.0168\nmAP@50                   : 0.6203\nmAP@50-95                : 0.3727\nTempo médio por imagem   : 0.9973 s\nConfiança média          : 0.7259\nErro médio de contagem   : 2.1096\n===================================================='

## 12. Exportação do modelo para ONNX (opcional)

Exporta os pesos para ONNX (usado pelo frontend web em `frontend/src/assets/modelo`).

In [13]:
# Exporta o YOLOv8m para ONNX (descomente para gerar o arquivo)
# modelo_export = YOLO(WEIGHTS)
# modelo_export.export(format="onnx")

## Conclusão

Este notebook organiza o projeto em funções reutilizáveis (`preparar_dataset`,
`carregar_modelo`, `contar_pessoas`, `avaliar_dataset`, `gerar_matriz_confusao`,
`gerar_graficos`, `salvar_relatorio`/`gerar_relatorio`) e produz, em `results/`, uma
tabela por imagem, gráficos, matriz de confusão e um relatório com Precision, Recall,
F1, mAP@50 e mAP@50-95 — tudo a partir de um único `dataset.zip`, sem versionar imagens.

> **Nota metodológica:** Precision/Recall/F1 são derivados do casamento por IoU
> (coerentes com a matriz de confusão e com a tabela de contagem), enquanto
> mAP@50 e mAP@50-95 vêm da validação oficial da ultralytics (`model.val`),
> que varre múltiplos limiares de confiança. As duas origens medem aspectos
> complementares do desempenho do detector.